In [0]:
display(dbutils.fs.ls("abfss://tgt-files@stcreditriskanalysis23.dfs.core.windows.net/"))

path,name,size,modificationTime
abfss://tgt-files@stcreditriskanalysis23.dfs.core.windows.net/applicant_profiles.parquet,applicant_profiles.parquet,2153973,1783614710000
abfss://tgt-files@stcreditriskanalysis23.dfs.core.windows.net/credit_application.parquet,credit_application.parquet,3803770,1783615104000
abfss://tgt-files@stcreditriskanalysis23.dfs.core.windows.net/credit_history.parquet,credit_history.parquet,2195749,1783615586000
abfss://tgt-files@stcreditriskanalysis23.dfs.core.windows.net/economic_indicators.parquet,economic_indicators.parquet,1730790,1783615902000
abfss://tgt-files@stcreditriskanalysis23.dfs.core.windows.net/loan_details.parquet,loan_details.parquet,3747730,1783616094000


In [0]:
df = spark.read.parquet(
    "abfss://tgt-files@stcreditriskanalysis23.dfs.core.windows.net/applicant_profiles.parquet"
)

display(df)

applicant_id,gender,age,income,business_or_commercial,occupancy_type,construction_type,total_units,region
24890,Sex Not Available,25-34,1740.0,nob/c,pr,sb,1U,south
24891,Male,55-64,4980.0,b/c,pr,sb,1U,North
24892,Male,35-44,9480.0,nob/c,pr,sb,1U,south
24893,Male,45-54,11880.0,nob/c,pr,sb,1U,North
24894,Joint,25-34,10440.0,nob/c,pr,sb,1U,North
24895,Joint,35-44,10080.0,nob/c,pr,sb,1U,North
24896,Joint,55-64,5040.0,nob/c,pr,sb,1U,North
24897,Female,55-64,3780.0,nob/c,pr,sb,1U,North
24898,Joint,55-64,5580.0,nob/c,pr,sb,1U,central
24899,Sex Not Available,55-64,6720.0,nob/c,pr,sb,1U,south


In [0]:
df.printSchema()

root
 |-- applicant_id: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: string (nullable = true)
 |-- income: string (nullable = true)
 |-- business_or_commercial: string (nullable = true)
 |-- occupancy_type: string (nullable = true)
 |-- construction_type: string (nullable = true)
 |-- total_units: string (nullable = true)
 |-- region: string (nullable = true)



In [0]:
from pyspark.sql.functions import *

bronze_df = (
    df
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", input_file_name())
    .withColumn("batch_id", expr("uuid()"))
)

In [0]:
from pyspark.sql.functions import *

bronze_df = (
    spark.read.parquet(
        "abfss://tgt-files@stcreditriskanalysis23.dfs.core.windows.net/applicant_profiles.parquet"
    )
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
    .withColumn("batch_id", expr("uuid()"))
)

In [0]:
from pyspark.sql.functions import *

bronze_df = (
    df
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
    .withColumn("batch_id", expr("uuid()"))
    .withColumn("load_date", current_date())
)

In [0]:
from pyspark.sql.functions import *
from datetime import datetime
import uuid

CATALOG = "credit_risk_analysis_db"
SCHEMA = "bronze_sch"

BASE_PATH = "abfss://tgt-files@stcreditriskanalysis23.dfs.core.windows.net/"

In [0]:
source_files = {

"applicant_profiles":"applicant_profiles.parquet",

"credit_application":"credit_application.parquet",

"credit_history":"credit_history.parquet",

"loan_details":"loan_details.parquet",

"economic_indicators":"economic_indicators.parquet"

}

In [0]:
def bronze_load(table_name,file_name):

    print(f"Loading {table_name}")

    df = (
        spark.read
        .format("parquet")
        .load(BASE_PATH + file_name)
    )

    bronze_df = (
        df
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("source_file", col("_metadata.file_path"))
        .withColumn("batch_id", lit(str(uuid.uuid4())))
        .withColumn("load_date", current_date())
    )

    (
        bronze_df.write
        .format("delta")
        .mode("overwrite")
        .option("mergeSchema","true")
        .saveAsTable(
            f"{CATALOG}.{SCHEMA}.bronze_{table_name}"
        )
    )

    print(f"{table_name} loaded successfully")

In [0]:
for table,file in source_files.items():

    bronze_load(table,file)

Loading applicant_profiles
applicant_profiles loaded successfully
Loading credit_application
credit_application loaded successfully
Loading credit_history
credit_history loaded successfully
Loading loan_details
loan_details loaded successfully
Loading economic_indicators
economic_indicators loaded successfully
